# Домашнее задание: Введение в LLM
**Занятие 41 | Неделя 21**

##Задание

Выполните все ячейки с пометкой `# ВАШ КОД ЗДЕСЬ`. Ответьте на теоретические вопросы в markdown-ячейках.

**Дедлайн:** до следующего занятия. Загрузите ноутбук в LMS.

**Критерии:**
- Код работает на Colab T4 GPU
- Все ячейки выполнены, результаты видны
- Теоретические вопросы отвечены своими словами (не копипаст)

## 0. Подготовка
Runtime -> Change runtime type -> **T4 GPU**

In [1]:
import torch
import time

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    print(f'GPU: {device_name}')
    device = torch.device('cuda')
else:
    print('CUDA is not available. Using CPU.')
    device = torch.device('cpu')

print(f'Device: {device}')

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Device: cuda


In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.6 MB/s eta 0:00:00


In [3]:
import numpy as np
from transformers import pipeline, AutoTokenizer

## Задание 1: Токенизация (Слайд 9)
Загрузите токенизатор модели `Qwen/Qwen2.5-0.5B-Instruct`. Токенизируйте 10 предложений: по 3-4 на каждом языке (русский, казахский, английский). Выведите для каждого: текст, количество токенов, сами токены. Постройте таблицу: язык, среднее кол-во токенов, среднее кол-во символов, отношение.

In [5]:
import pandas as pd

# Сначала загружаем токенизатор
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

sentences = [
    "The weather is beautiful today.", "Learning AI is very exciting.", "I love coding in Python.",
    "Сегодня на улице прекрасная погода.", "Изучать ИИ очень увлекательно.", "Мне нравится программировать.",
    "Бүгін күн райы тамаша.", "Жасанды интеллектті үйрену қызық.", "Маған код жазу ұнайды.", "Астана - әдемі қала."
]

data = []
for text in sentences:
    tokens = tokenizer.encode(text)
    token_strings = tokenizer.convert_ids_to_tokens(tokens)

    # Определяем язык для статистики
    if any(char in text.lower() for char in 'abcdefghijklmnopqrstuvwxyz') and 'th' in text.lower():
        lang = 'en'
    elif any(char in text.lower() for char in 'йцукенгшщзхъфывапролджэячсмитьбю') and not any(char in text.lower() for char in 'әғқңөұүһі'):
        lang = 'ru'
    else:
        lang = 'kz'

    data.append({
        "lang": lang,
        "text": text,
        "tokens_count": len(tokens),
        "tokens": token_strings,
        "chars_count": len(text)
    })
    print(f"Текст: {text}\nКол-во токенов: {len(tokens)}\nТокены: {token_strings}\n")

# Таблица статистики
df = pd.DataFrame(data)
stats = df.groupby("lang").agg({
    "tokens_count": "mean",
    "chars_count": "mean"
})
stats["ratio (chars/tokens)"] = stats["chars_count"] / stats["tokens_count"]
print(stats)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Текст: The weather is beautiful today.
Кол-во токенов: 6
Токены: ['The', 'Ġweather', 'Ġis', 'Ġbeautiful', 'Ġtoday', '.']

Текст: Learning AI is very exciting.
Кол-во токенов: 6
Токены: ['Learning', 'ĠAI', 'Ġis', 'Ġvery', 'Ġexciting', '.']

Текст: I love coding in Python.
Кол-во токенов: 6
Токены: ['I', 'Ġlove', 'Ġcoding', 'Ġin', 'ĠPython', '.']

Текст: Сегодня на улице прекрасная погода.
Кол-во токенов: 12
Токены: ['Ð¡', 'ÐµÐ³Ð¾Ð´Ð½Ñı', 'ĠÐ½Ð°', 'ĠÑĥ', 'Ð»Ð¸ÑĨ', 'Ðµ', 'ĠÐ¿ÑĢÐµÐºÑĢÐ°Ñģ', 'Ð½Ð°Ñı', 'ĠÐ¿', 'Ð¾Ð³', 'Ð¾Ð´Ð°', '.']

Текст: Изучать ИИ очень увлекательно.
Кол-во токенов: 11
Токены: ['Ðĺ', 'Ð·', 'ÑĥÑĩ', 'Ð°ÑĤÑĮ', 'ĠÐĺ', 'Ðĺ', 'ĠÐ¾ÑĩÐµÐ½ÑĮ', 'ĠÑĥ', 'Ð²Ð»ÐµÐºÐ°ÑĤÐµÐ»ÑĮ', 'Ð½Ð¾', '.']

Текст: Мне нравится программировать.
Кол-во токенов: 8
Токены: ['Ðľ', 'Ð½Ðµ', 'ĠÐ½', 'ÑĢÐ°Ð²', 'Ð¸ÑĤÑģÑı', 'ĠÐ¿ÑĢÐ¾Ð³ÑĢÐ°Ð¼Ð¼', 'Ð¸ÑĢÐ¾Ð²Ð°ÑĤÑĮ', '.']

Текст: Бүгін күн райы тамаша.
Кол-во токенов: 14
Токены: ['Ðĳ', 'Ò¯', 'Ð³', 'Ñĸ', 'Ð½', 'ĠÐº', 'Ò¯', 'Ð½', 'ĠÑĢÐ°Ð¹', 'Ñĭ', 'ĠÑĤÐ°Ð¼', 'Ð°ÑĪ', 'Ð°',

**Вопрос 1.1 (Слайд 9):** Почему казахский и русский текст требуют больше токенов, чем английский? Как это влияет на стоимость использования LLM?

*Ваш ответ:* Большинство токенизаторов обучаются на английских текстах, поэтому английские слова делятся на целые слова, а русские и казахские на мелкие слоги или даже буквы. Это увеличивает стоимость, так как оплата в API идет за количество токенов

**Вопрос 1.2 (Слайд 9):** Что такое BPE (Byte Pair Encoding)? Опишите алгоритм в 3-4 шагах своими словами.

*Ваш ответ:* BPE - это алгоритм сжатия текста в токены.
1. Начинаем с отдельных букв.
2. Ищем самую частую пару символов рядом.
3. Объединяем их в один новый токен.
4. Повторяем, пока не достигнем нужного размера словаря.

## Задание 2: Генерация текста (Слайды 8, 13)
Загрузите модель через pipeline и сгенерируйте текст с разными параметрами.

In [7]:

generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.float16,
)

prompt = "Расскажи интересный факт об Астане"
temperatures = [0.1, 0.5, 0.7, 1.0, 1.5]

for temp in temperatures:
    sampling = True if temp > 0.1 else False

    res = generator(
        prompt,
        max_new_tokens=50,
        temperature=temp,
        do_sample=sampling,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    print(f"Температура: {temp}")
    print(f"Ответ: {res[0]['generated_text']}")
    print("-" * 30)

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'pad_token_id', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Температура: 0.5
Ответ: Расскажи интересный факт об Астане, которая может быть полезен для путешественников. Астрахань - это крупнейший город в Казахстане и столица страны. В 1840 году он был основан монархией
------------------------------


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Температура: 0.7
Ответ: Расскажи интересный факт об Астане. - 185024
185024: Семь лет в Астане
1850-51 года, когда Россия проводила военные кампании по захвату А
------------------------------


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Температура: 1.0
Ответ: Расскажи интересный факт об Астане - Астана - Великая Россия
Главная страница > Новости > Культура и искусство > Астана
Согласно данным ведомства, Астана в 2019 году
------------------------------
Температура: 1.5
Ответ: Расскажи интересный факт об Астане

Крымское монастырство Майерский, возвышающееся над рекой Дюшеву (ныне рекой Азы) в Станице Казачихи на границе
------------------------------


**Вопрос 2.1 (Слайд 13):** Как температура влияет на генерацию? При какой температуре ответы наиболее стабильные? При какой — начинается "мусор"?

*Ваш ответ:* Температура меняет случайность: низкая делает ответы предсказуемыми, высокая — креативными. Стабильные ответы при 0.1–0.5. "Мусор"  начинается выше 1.2.

In [8]:
prompt_it = "Придумай слоган для казахстанского IT-университета"

print("--- T=0 ---")
for _ in range(5):
    res = generator(prompt_it, max_new_tokens=30, temperature=0, do_sample=False)
    print(res[0]['generated_text'])

print("\n--- T=0.9 ---")
for _ in range(5):
    res = generator(prompt_it, max_new_tokens=30, temperature=0.9, do_sample=True)
    print(res[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- T=0 ---


Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Придумай слоган для казахстанского IT-университета "Казахстанский Институт Технологий". 

Вот список ключевых слов и их значения:

1. Каз


Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Придумай слоган для казахстанского IT-университета

Король Исламова

Маленький Исполнительный секретарь

Начальник по развитию и управ


Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Придумай слоган для казахстанского IT-университета. Используйте какую-то из известных фраз из биографии и исторических событий, чтобы привлеч


Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Придумай слоган для казахстанского IT-университета "Салтак".

Для этого нужно использовать слово "Салтак" как основу и придумать уникальный сино
Придумай слоган для казахстанского IT-университета.

Конечно, вот несколько вариантов слогана для казахстанского IT-университета:

1. "И


**Вопрос 2.2 (Слайд 8):** Что такое autoregressive генерация? Почему модель генерирует токен за токеном, а не весь текст сразу?

*Ваш ответ:* Это генерация где каждый новый токен зависит от всех предыдущих. Модель не знает конца фразы, пока не предскажет все слова по очереди, опираясь на контекст.

## Задание 3: Zero-shot vs Few-shot (Слайд 14)
Реализуйте классификатор тональности отзывов двумя способами: zero-shot (без примеров) и few-shot (с примерами в промпте).

In [10]:
test_reviews = [
    "Все было идеально, спасибо большое!",
    "Никогда больше не вернусь в это место.",
    "Обычное кафе, ничего выдающегося.",
    "Быстрая доставка, товар как на фото.",
    "Обман чистой воды, не заказывайте.",
    "Сойдет, но за такую цену ожидал большего.",
]

def classify_zero_shot(text):
    messages = [
        {"role": "system", "content": "Ты классификатор тональности. Отвечай только одним словом: позитивный, негативный или нейтральный."},
        {"role": "user", "content": text}
    ]
    res = generator(messages, max_new_tokens=10)
    return res[0]['generated_text'][-1]['content'].strip()

for rev in test_reviews:
    result = classify_zero_shot(rev)
    print(f"Отзыв: {rev} -> {result}")

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Все было идеально, спасибо большое! -> позитивный


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Никогда больше не вернусь в это место. -> Нейтральный


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Обычное кафе, ничего выдающегося. -> негативный


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Быстрая доставка, товар как на фото. -> Позитивный


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Обман чистой воды, не заказывайте. -> негативный
Отзыв: Сойдет, но за такую цену ожидал большего. -> negativity


In [11]:
def classify_few_shot(text):
    prompt = """Классифицируй отзыв.
Пример 1: Ужасная еда. Ответ: негативный
Пример 2: Принесли быстро, вкусно. Ответ: позитивный
Пример 3: Обычный магазин. Ответ: нейтральный
Пример 4: Сломалось через день. Ответ: негативный
Отзыв: """ + text + "\nОтвет:"
    res = generator(prompt, max_new_tokens=10)
    return res[0]['generated_text'].split("Ответ:")[-1].strip()

for rev in test_reviews:
    print(f"Отзыв: {rev} -> {classify_few_shot(rev)}")

Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Все было идеально, спасибо большое! -> позитивный

Ваш ответ на


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Никогда больше не вернусь в это место. -> негативный

Сравните


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Обычное кафе, ничего выдающегося. -> негативный

Ваш ответ на


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Быстрая доставка, товар как на фото. -> негативный

Вопрос:


Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Отзыв: Обман чистой воды, не заказывайте. -> негативный

Вот ваши отзывы
Отзыв: Сойдет, но за такую цену ожидал большего. -> негативный

Вот несколько пример


**Вопрос 3.1 (Слайд 14):** В чем разница между zero-shot и few-shot? Когда few-shot дает лучшие результаты?

*Ваш ответ:* Zero-shot - это работа без примеров, только по инструкции. Few-shot включает примеры в промпт. Few-shot лучше работает в сложных задачах, где модели нужно показать нужный формат или стиль ответа.

## Задание 4: System Prompt (Слайд 14)
Напишите 3 разных system prompt для одной модели, чтобы она вела себя как:
1. Учитель математики
2. Переводчик каз-рус
3. Ваша идея (придумайте свой вариант)

In [12]:
roles = {
    "Учитель": "Ты строгий учитель математики. Объясняй кратко и давай задание.",
    "Переводчик": "Ты переводчик. Переводи с казахского на русский и наоборот без лишних слов.",
    "Зумер": "Ты подросток-зумер. Используй сленг, пиши без заглавных букв."
}

tests = ["Сколько будет 2+2?", "Как дела?"]

for role, sys_text in roles.items():
    print(f"--- Роль: {role} ---")
    for q in tests:
        msg = [{"role": "system", "content": sys_text}, {"role": "user", "content": q}]
        res = generator(msg, max_new_tokens=50)
        print(f"Вопрос: {q} | Ответ: {res[0]['generated_text'][-1]['content']}")

Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Роль: Учитель ---


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Вопрос: Сколько будет 2+2? | Ответ: 2+2 равно 4.


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Вопрос: Как дела? | Ответ: Хорошо! Как я могу помочь вам сегодня?
--- Роль: Переводчик ---


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Вопрос: Сколько будет 2+2? | Ответ: 2+2 равно 4.


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Вопрос: Как дела? | Ответ: Хорошо! Как я могу помочь вам сегодня?
--- Роль: Зумер ---
Вопрос: Сколько будет 2+2? | Ответ: 4
Вопрос: Как дела? | Ответ: Хорошо! Как я могу помочь вам сегодня?


**Вопрос 4.1 (Слайд 14):** Зачем нужен system prompt? Чем он отличается от обычного user-сообщения?

*Ваш ответ:* System prompt задает общие правила поведения, тон и ограничения для модели на всю сессию. Обычное сообщение - это разовый вопрос пользователя в рамках этих правил.

## Задание 5: Теоретические вопросы (по слайдам)
Ответьте на вопросы своими словами. Не копируйте — объясните как вы понимаете.

**Вопрос 5.1 (Слайд 6):** Чем отличается Self-Attention в BERT (bidirectional) от Causal Attention в GPT (unidirectional)? Нарисуйте или опишите маску внимания для обоих случаев.

*Ваш ответ:* Self-Attention видит все слова сразу и слева, и справа, маска пустая. Causal Attention видит только слова слева, закрывая будущие слова маской, чтобы модель не подглядывала в ответ

**Вопрос 5.2 (Слайд 7):** Назовите 3 типа архитектур LLM (encoder-only, decoder-only, encoder-decoder). Для каждого приведите пример модели и задачу, для которой он подходит.

*Ваш ответ:*
1. Encoder-only: BERT (классификация текста).
2. Decoder-only: GPT-4 (генерация чат-ботов).
3. Encoder-decoder: T5 (перевод текста).

**Вопрос 5.3 (Слайд 11):** Что значит "открытая модель"? Объясните разницу между API-only, Open Weights и Open Source. Почему открытые модели важны для Казахстана?

*Ваш ответ:* Open Weights - веса модели доступны для скачивания. API-only - доступ только через сайт. Open Source - открыт еще и код обучения. Для Казахстана это важно для независимости от санкций и защиты данных.

**Вопрос 5.4 (Слайд 12):** Что такое квантизация? Зачем она нужна? Сколько VRAM нужно для LLaMA 3 8B в FP16 и в INT4?

*Ваш ответ:* Квантизация - это снижение точности чисел весов для экономии памяти. Llama 3 8B: FP16 требует ~16 ГБ VRAM, INT4 требует ~5.5 ГБ VRAM.

**Вопрос 5.5 (Слайд 16):** Назовите 3 ограничения LLM. Для каждого приведите конкретный пример, когда это может быть проблемой.

*Ваш ответ:*
1. Галлюцинации: может придумать закон, которого нет.
2. Окно контекста: забывает начало длинной книги.
3. Устаревание знаний: не знает о событиях, случившихся после ее обучения

## Чеклист перед сдачей
- [ ] Задание 1: токенизация 10 предложений + таблица
- [ ] Задание 2: генерация при 5 температурах + анализ
- [ ] Задание 3: zero-shot и few-shot классификация + сравнение
- [ ] Задание 4: 3 system prompt + тестовые запросы
- [ ] Задание 5: все 5 теоретических вопросов отвечены
- [ ] Все вопросы отвечены своими словами
- [ ] Ноутбук запускается с нуля (Runtime -> Restart and Run All)

**Загрузите ноутбук в LMS.**